# Tier 3 cross-encoder: train, tune the decoder on dev, export

RUN_DECISIONS D19, "tier 3 design". Disjoint corpus only; this notebook and its bundle contain no evaluation pool.

Needs in `My Drive/fragmentation_t3/`: `t3_code_bundle.zip`, `t3_training_data.zip`. Runtime: GPU (A100 preferred). Re-running after a disconnect resumes from the last checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, subprocess, json
BASE = '/content/drive/MyDrive/fragmentation_t3'
OUT = BASE + '/out'
os.makedirs(OUT, exist_ok=True)
for z, dst in (('t3_code_bundle.zip', '/content/t3'), ('t3_training_data.zip', '/content/t3data')):
    assert os.path.exists(f'{BASE}/{z}'), f'{z} is not in {BASE}'
    zipfile.ZipFile(f'{BASE}/{z}').extractall(dst)
print('bundle commit:', open('/content/t3/BUNDLE_COMMIT.txt').read().strip())
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'NO GPU: set Runtime > Change runtime type > GPU')

In [ ]:
!pip -q install python-dotenv openai onnx onnxruntime
!pip list 2>/dev/null | grep -i -E '^(torch|transformers|tokenizers|onnx|onnxruntime|safetensors|scikit-learn|scipy|numpy) '

In [ ]:
!cd /content/t3 && python -u runs/t3_train.py --data /content/t3data/documents.jsonl --out "$OUT"

In [ ]:
!cd /content/t3 && python -u runs/t3_dev.py --data /content/t3data/documents.jsonl --out "$OUT"

In [ ]:
rec = json.load(open(OUT + '/RECORD.json'))
print(json.dumps({k: rec[k] for k in ('model_sha256', 'onnx_sha256', 'tokenizer_sha256', 'chosen', 'dev_T3', 'bundle_commit')}, indent=1))
print('T3 COLAB RUN COMPLETE')
!ls -la "$OUT" "$OUT/model"